# Notebook 9: Presentation Recording Demo

This notebook is a single-image AADS v6 demo designed for screen recording. It runs the real router and adapter flow, then presents SAM3 region proposals, BioCLIP-2.5 crop/part evidence, the safety gate, adapter handoff, and the disease/OOD result in one readable panel.

**Recording note:** SAM3 boxes mark candidate plant regions for inspection. They are not disease predictions.

## Usage

1. Select a Colab GPU runtime.
2. Run the first three setup cells once.
3. Run the Preview and Warm-up cell once and upload the final demo image.
4. Start screen recording and run only the final rendering cell.
5. For a new image, rerun Preview and Warm-up before recording again.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
os.environ['AADS_COLAB_REQUIREMENTS_FILE'] = 'requirements_presentation_colab.txt'


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if CLONE_TARGET.exists():
        shutil.rmtree(CLONE_TARGET)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    repo_root = CLONE_TARGET.resolve()
    marker = repo_root / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
    if not marker.is_file():
        raise FileNotFoundError(f'Notebook bootstrap could not find the cell runner under {repo_root}.')
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    return repo_root


print('[SETUP] Preparing the presentation demo...', flush=True)
_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell01_bootstrap.py', globals())
print('[SETUP] Presentation demo setup complete. Continue to the next cell.', flush=True)


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell02_access_check.py', globals())

from IPython.display import clear_output
clear_output(wait=True)


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Keep detailed router outputs enabled for the presentation recording.
NOTEBOOK_PROFILE = 'a100_colab_default'
NOTEBOOK_SPEED_MODE = 'balanced'
INFERENCE_OVERRIDES = {
    'SHOW_ROUTER_DIAGNOSTICS': True,
    'TOP_CROP_CANDIDATES': 5,
    'RENDER_ROUTER_VISUALIZATION': False,
    'INCLUDE_ADAPTER_TARGET': True,
}
ADAPTER_ROOT = None
RETURN_OOD = True
PRINT_JSON_RESULT = False

run_cell_script('nb1_cell03_runtime_setup.py', globals())

from IPython.display import clear_output
clear_output(wait=True)


## Preview and Warm-up

Run this cell before recording. Upload the final demo image once. The real router, adapter, and OOD flow runs here so the recording cell can render the result immediately.

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from IPython.display import clear_output

ANALYSIS_IMAGE_PATH = None  # Upload the final recording image.

technical_output = StringIO()
with redirect_stdout(technical_output), redirect_stderr(technical_output):
    run_cell_script('nb1_cell04_analysis.py', globals())
    router_result = result
    run_cell_script('nb8_cell05_adapter_prediction.py', globals())

clear_output(wait=True)
print('Preview ready. Start recording and run the final cell.')


## Recording Cell

Start screen recording and run this cell. It renders the warmed-up real inference result immediately.

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
from IPython.display import clear_output

if not globals().get('router_result') or not globals().get('auto_result'):
    raise RuntimeError('Run the Preview and Warm-up cell before recording.')
clear_output(wait=True)
run_cell_script('nb9_cell06_presentation_demo.py', globals())
